### Deliverable 4

In [25]:
import datetime
import json

import boto3
import requests

In [26]:
USERNAME = "aruzhan"

In [27]:
DATE_PARAM = "2025-11-25"

date = datetime.datetime.strptime(DATE_PARAM, "%Y-%m-%d")

# Construct the API URL
url = f"https://wikimedia.org/api/rest_v1/metrics/pageviews/top/en.wikipedia.org/all-access/{date.strftime('%Y/%m/%d')}"
print(f"Requesting REST API URL: {url}")

Requesting REST API URL: https://wikimedia.org/api/rest_v1/metrics/pageviews/top/en.wikipedia.org/all-access/2025/11/25


In [28]:
# Make the API request
wiki_server_response = requests.get(url, headers={"User-Agent": "curl/7.68.0"})
wiki_response_status = wiki_server_response.status_code
wiki_response_body = wiki_server_response.text

print(f"Wikipedia REST API Response body: {wiki_response_body[:500]}...")
print(f"Wikipedia REST API Response Code: {wiki_response_status}")

# Validate response
if wiki_response_status != 200:
    raise Exception(f"Received non-OK status code from Wiki Server: {wiki_response_status}")
print(f"Successfully retrieved Wikipedia data, content-length: {len(wiki_response_body)}")

Wikipedia REST API Response body: {"items":[{"project":"en.wikipedia","access":"all-access","year":"2025","month":"11","day":"25","articles":[{"article":"Main_Page","views":6343624,"rank":1},{"article":"Special:Search","views":867370,"rank":2},{"article":"Dharmendra","views":386721,"rank":3},{"article":"Sawyer_Sweeten","views":314428,"rank":4},{"article":"Google_Chrome","views":308826,"rank":5},{"article":"Wikipedia:Featured_pictures","views":259967,"rank":6},{"article":"Richard_Branson","views":160194,"rank":7},{"article":"Wick...
Wikipedia REST API Response Code: 200
Successfully retrieved Wikipedia data, content-length: 55589


In [29]:
# Parse the API response and extract top edits
wiki_response_parsed = wiki_server_response.json()
most_views = wiki_response_parsed["items"][0]["articles"]

# Transform to JSON Lines format
current_time = datetime.datetime.now(datetime.timezone.utc)
json_lines = ""
for page in most_views[:5]:
    record = {
        "article": page["article"],
        "views": page["views"],
        "rank": page["rank"],
        "date": date.strftime("%Y-%m-%d"),
        "retrieved_at": current_time.replace(tzinfo=None).isoformat(),
    }
    json_lines += json.dumps(record) + "\n"

print(f"Transformed {len(most_views)} records to JSON Lines")
print(f"First few lines:\n{json_lines[:500]}...")

Transformed 1000 records to JSON Lines
First few lines:
{"article": "Main_Page", "views": 6343624, "rank": 1, "date": "2025-11-25", "retrieved_at": "2025-12-10T16:31:16.257440"}
{"article": "Special:Search", "views": 867370, "rank": 2, "date": "2025-11-25", "retrieved_at": "2025-12-10T16:31:16.257440"}
{"article": "Dharmendra", "views": 386721, "rank": 3, "date": "2025-11-25", "retrieved_at": "2025-12-10T16:31:16.257440"}
{"article": "Sawyer_Sweeten", "views": 314428, "rank": 4, "date": "2025-11-25", "retrieved_at": "2025-12-10T16:31:16.257440"}
{"ar...


In [30]:
S3_WIKI_BUCKET = "aruzhan-wikidata"
s3 = boto3.client("s3")
default_region = 'eu-west-1'
bucket_names = [bucket["Name"] for bucket in s3.list_buckets()["Buckets"]]
if S3_WIKI_BUCKET not in bucket_names:
    bucket_configuration = {"LocationConstraint": default_region}
    response = s3.create_bucket(Bucket=S3_WIKI_BUCKET, CreateBucketConfiguration=bucket_configuration)
    print(f"Created new bucket: {S3_WIKI_BUCKET}")
else:
    print(f"Using existing bucket: {S3_WIKI_BUCKET}")


Using existing bucket: aruzhan-wikidata


In [31]:
# Test Lab 1
assert USERNAME != "<username>", "Please set your USERNAME at the top of the notebook"
assert S3_WIKI_BUCKET.endswith("-wikidata"), "Bucket name must end with '-wikidata'"

try:
    s3.head_bucket(Bucket=S3_WIKI_BUCKET)
    print(f"Bucket {S3_WIKI_BUCKET} exists!")
except Exception as e:
    print(f"Bucket {S3_WIKI_BUCKET} not found: {e}")
    raise

Bucket aruzhan-wikidata exists!


In [32]:
# Lab 2 Solution: Upload json_lines directly to S3
s3_key = f"raw-views/raw-views-{date.strftime('%Y-%m-%d')}.json"
s3.put_object(
    Bucket=S3_WIKI_BUCKET,
    Key=s3_key,
    Body=json_lines,
)
print(f"Uploaded {len(most_views)} records to s3://{S3_WIKI_BUCKET}/{s3_key}")

Uploaded 1000 records to s3://aruzhan-wikidata/raw-views/raw-views-2025-11-25.json


In [33]:
expected_key = f"raw-views/raw-views-{date.strftime('%Y-%m-%d')}.json"
try:
    s3.head_object(Bucket=S3_WIKI_BUCKET, Key=expected_key)
    print(f"File uploaded successfully to s3://{S3_WIKI_BUCKET}/{expected_key}")
except Exception as e:
    print(f"File not found at s3://{S3_WIKI_BUCKET}/{expected_key}")
    raise

File uploaded successfully to s3://aruzhan-wikidata/raw-views/raw-views-2025-11-25.json
